In [ ]:
pip install torchmetrics

-----------------------------------------

In [1]:
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import Audio, clear_output
import random 
import matplotlib.pyplot as plt
from torchmetrics.functional.classification import multilabel_accuracy,multilabel_f1_score,multiclass_accuracy,multiclass_f1_score
from torchmetrics.functional import accuracy,f1_score


In [2]:
import torch
import torch.nn as nn
from torch.amp import autocast,GradScaler
# import torch_xla

# import torch_xla.core.xla_model as xm
from torch.utils.data import DataLoader,Dataset,random_split
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim.lr_scheduler import CosineAnnealingLR,ReduceLROnPlateau

from transformers import ASTFeatureExtractor, ASTModel, ASTConfig, AutoModelForAudioClassification

In [3]:
extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
astmodel = AutoModelForAudioClassification.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593",attn_implementation="sdpa")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [4]:
# device=xm.xla_device()

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
random.seed(42)

## Dataset Processing

In [9]:
sr = 16000

In [10]:
def load_audio(file_path, sr=None, duration=None):
    audio, orig = librosa.load(file_path)
    # Resample the audio
    if orig != 16000:
        audio = librosa.resample(audio, orig_sr=orig, target_sr=16000)
#     audio, _ = librosa.load(file_path,sr=16000)
    return audio

In [11]:
def mix(z,df,num_labels,path):
    audio = load_audio(path + df.iloc[z[0]]['filename'])
    a = torch.zeros(num_labels)
    a[df.iloc[z[0]]["target"]] = 1

        
    return audio,a

In [12]:
def time_stretching(rate, sound_clip):
  sound_clip = librosa.effects.time_stretch(y=sound_clip, rate = 16000)
  return sound_clip

In [13]:
def pitch_shifting(tone_step, sound_clip, sr=16000):
  sound_clip = librosa.effects.pitch_shift(y=sound_clip, sr=sr, n_steps = tone_step)
  return sound_clip

In [14]:
def add_noise(sound_clip):
    noise = np.random.rand(len(sound_clip))
#     noise = np.ones(len(sound_clip))
    noise_amp = np.random.uniform(0.005, 0.008)
    noisy_sound_clip = sound_clip + (noise_amp * noise)
    return noisy_sound_clip

In [15]:
def stretch(data, rate=1):
    input_length = len(data)
    data = librosa.effects.time_stretch(y=data,rate=rate)
    if len(data)>input_length:
        data = data[:input_length]
    else:
        data = np.pad(data, (0, max(0, input_length - len(data))), "constant")

    return data

In [16]:
y = load_audio("/kaggle/input/environmental-sound-classification-50/audio/audio/16000/1-101296-B-19.wav")

In [17]:
class SoundDS(Dataset):
    
    def __init__(self, df, extractor, data_path, sr, file_list,num_labels):
        self.df = df
        self.data_path = data_path
        self.sr = sr
        self.file_list = file_list
        self.extractor = extractor
    
    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        
        x,c = self.file_list[idx]
        
        y,a = mix(x,self.df,num_labels,path=self.data_path)
        if c == 0:
            y = librosa.util.normalize(y)
            f = self.extractor(y,sampling_rate=self.sr, padding="max_length", return_tensors="pt").input_values[0]
            return f,a
        
        if c == 3:
            y1 = stretch(y,0.8)
            y1 = librosa.util.normalize(y1)
            f = self.extractor(y1,sampling_rate=self.sr, padding="max_length", return_tensors="pt").input_values[0]        
            return f,a
        
        if c == 1:
            y1 = pitch_shifting(2, y)
            y1 = librosa.util.normalize(y1)
            f = self.extractor(y1,sampling_rate=self.sr, padding="max_length", return_tensors="pt").input_values[0]     
            return f,a
        
        if c == 2:
            y1 = add_noise(y)
            y1 = librosa.util.normalize(y1)
            f = self.extractor(y1,sampling_rate=self.sr, padding="max_length", return_tensors="pt").input_values[0]    
            return f,a
        
        if c == 4:
            y1 = pitch_shifting(-2,y)
            y1 = librosa.util.normalize(y1)
            f = self.extractor(y1,sampling_rate=self.sr, padding="max_length", return_tensors="pt").input_values[0]    
            return f,a
        
        if c == 5:
            y1 = stretch(y,1.2)
            y1 = librosa.util.normalize(y1)
            f = self.extractor(y1,sampling_rate=self.sr, padding="max_length", return_tensors="pt").input_values[0]    
            return f,a

In [18]:
device = "cuda"

## Model Definition

In [19]:
class MultiAudioClassifier(nn.Module):
    def __init__(self, num_labels, lstm_hidden_size=512, num_lstm_layers=1):
        super(MultiAudioClassifier, self).__init__()
        
        self.model = AutoModelForAudioClassification.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593",
            attn_implementation="sdpa"
        )
        
        self.pretrained_output_size = 527
        
        for param in self.model.parameters():
            param.requires_grad = False
        
        for param in self.model.classifier.parameters():
            param.requires_grad = True
        
        self.lstm = nn.LSTM(
            input_size=self.pretrained_output_size,
            hidden_size=lstm_hidden_size,
            num_layers=num_lstm_layers,
            bidirectional=True,
        )
        
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden_size * 2, 256),
            nn.LeakyReLU(),
            nn.BatchNorm1d(256, momentum=0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(),
            nn.Dropout(0.3), 
            nn.Linear(128, num_labels)
        )
    
    def forward(self, features):
        transformer_output = self.model(features).logits
        
        lstm_output, _ = self.lstm(transformer_output.unsqueeze(1))
        
        lstm_last_output = lstm_output.squeeze(1)
        
        output = self.fc(lstm_last_output)
        
        return output

## Training Function

In [20]:
device = "cuda"

In [23]:
import torch
from torchmetrics.functional import accuracy, f1_score, precision, recall, confusion_matrix

def calculate_metrics(y_pred_logits, y_true_onehot, num_classes):

    y_true_indices = torch.argmax(y_true_onehot, dim=1)
    
    acc = accuracy(y_pred_logits, y_true_indices, task='multiclass', num_classes=num_classes)
    f1_macro = f1_score(y_pred_logits, y_true_indices, task='multiclass', num_classes=num_classes, average='macro')
    f1_weighted = f1_score(y_pred_logits, y_true_indices, task='multiclass', num_classes=num_classes, average='weighted')
    f1_micro = f1_score(y_pred_logits, y_true_indices, task='multiclass', num_classes=num_classes, average='micro')
    prec = precision(y_pred_logits, y_true_indices, task='multiclass', num_classes=num_classes)
    rec = recall(y_pred_logits, y_true_indices, task='multiclass', num_classes=num_classes)
    


    return acc.item(), f1_macro.item(), f1_micro.item(), f1_weighted.item(), prec.item(), rec.item()
    

In [ ]:
CLA_label = {0: 'dog', 14: 'chirping_birds', 36: 'vacuum_cleaner', 19: 'thunderstorm', 30: 'door_wood_knock',34: 'can_opening', 9: 'crow', 22: 'clapping', 48: 'fireworks', 41: 'chainsaw', 47: 'airplane', 31: 'mouse_click', 17: 'pouring_water', 45: 'train', 8: 'sheep', 15: 'water_drops', 46: 'church_bells', 37: 'clock_alarm', 32: 'keyboard_typing', 16: 'wind', 25: 'footsteps', 4: 'frog', 3: 'cow', 27: 'brushing_teeth', 43: 'car_horn', 12: 'crackling_fire', 40: 'helicopter', 29: 'drinking_sipping', 10: 'rain', 7: 'insects', 26: 'laughing', 6: 'hen', 44: 'engine', 23: 'breathing', 20: 'crying_baby', 49: 'hand_saw', 24: 'coughing', 39: 'glass_breaking', 28: 'snoring', 18: 'toilet_flush', 2: 'pig', 35: 'washing_machine', 38: 'clock_tick', 21: 'sneezing', 1: 'rooster', 11: 'sea_waves', 42: 'siren', 5: 'cat', 33: 'door_wood_creaks', 13: 'crickets'}

In [25]:
def train_epoch(model, train_dl, criterion, optimizer, scaler, device, num_labels):
    model.train()
    train_loss = 0
    train_acc = 0
    train_f1_micro = 0
    train_f1_macro = 0
    train_f1_weighted = 0
    train_prec = 0
    train_rec = 0
    num_batches = len(train_dl)

    pbar = tqdm(train_dl, desc='Training')
    for step, (x, y) in enumerate(pbar):
        x, y = x.squeeze().to(device, non_blocking=True), y.squeeze().to(device, non_blocking=True)
        
        with autocast(device_type="cuda",enabled=True):
            p = model(x)
            loss = criterion(p, y)
        
        scaler.scale(loss).backward()
        
        if (step + 1) % 4 == 0:  # Gradient accumulation for 4 steps
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        train_loss += loss.item()

        with torch.no_grad():
            acc, f1_mac, f1_mic, f1_wei, prec, rec = calculate_metrics(p, y,num_classes=num_labels)
            train_acc += acc
            train_f1_micro += f1_mic
            train_f1_macro += f1_mac
            train_f1_weighted += f1_wei
            train_prec += prec
            train_rec += rec

        pbar.set_postfix({
            'Loss': f'{train_loss/(step+1):.4f}', 
            'Acc': f'{train_acc/(step+1):.4f}', 
            'Macro-F1': f'{train_f1_macro/(step+1):.4f}', 
            'Micro-F1': f'{train_f1_micro/(step+1):.4f}',
            'Weighted-F1': f'{train_f1_weighted/(step+1):.4f}',
            'Precision': f'{train_prec/(step+1):.4f}',
            'Recall': f'{train_rec/(step+1):.4f}'
        })

    avg_loss = train_loss / num_batches
    avg_acc = train_acc / num_batches
    avg_f1_macro = train_f1_macro / num_batches
    avg_f1_micro = train_f1_micro / num_batches
    avg_f1_weighted = train_f1_weighted / num_batches
    avg_prec = train_prec / num_batches
    avg_rec = train_rec / num_batches

    return avg_loss, avg_acc, avg_f1_macro, avg_f1_micro, avg_f1_weighted, avg_prec, avg_rec

def validate(model, val_dl, criterion, device, num_labels):
    model.eval()
    val_loss = 0
    val_acc = 0
    val_f1_micro = 0
    val_f1_macro = 0
    val_f1_weighted = 0
    val_prec = 0
    val_rec = 0
    num_batches = len(val_dl)
    
    with torch.no_grad():
        for x, y in tqdm(val_dl, desc='Validation'):
            x, y = x.squeeze().to(device, non_blocking=True), y.squeeze().to(device, non_blocking=True)
            
            with autocast(device_type="cuda", enabled=True):
                p = model(x)
                loss = criterion(p, y)
            
            val_loss += loss.item()
            acc, f1_mac, f1_mic, f1_wei, prec, rec = calculate_metrics(p, y, num_classes=num_labels)
            
            val_acc += acc
            val_f1_macro += f1_mac
            val_f1_micro += f1_mic
            val_f1_weighted += f1_wei
            val_prec += prec
            val_rec += rec

    avg_loss = val_loss / num_batches
    avg_acc = val_acc / num_batches
    avg_f1_macro = val_f1_macro / num_batches
    avg_f1_micro = val_f1_micro / num_batches
    avg_f1_weighted = val_f1_weighted / num_batches
    avg_prec = val_prec / num_batches
    avg_rec = val_rec / num_batches

    return avg_loss, avg_acc, avg_f1_macro, avg_f1_micro, avg_f1_weighted, avg_prec, avg_rec

def train(model, train_dl, val_dl, criterion, optimizer, num_epochs, device, num_labels):
    scaler = GradScaler()
    val_loss, val_acc, val_f1_macro, val_f1_micro, val_f1_weighted, val_prec, val_rec =  0,0,0,0,0,0,0
    # Initialize metrics to track across all epochs
    best_val = 0
    c = 0
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        
        # Training for one epoch
        train_loss, train_acc, train_f1_macro, train_f1_micro, train_f1_weighted, train_prec, train_rec = train_epoch(
            model, train_dl, criterion, optimizer, scaler, device, num_labels)
        
        # Validation
        val_loss, val_acc, val_f1_macro, val_f1_micro, val_f1_weighted, val_prec, val_rec = validate(
            model, val_dl, criterion, device, num_labels)
        
        # Update metrics
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Train Macro-F1: {train_f1_macro:.4f}, '
              f'Train Micro-F1: {train_f1_micro:.4f}, Train Weighted-F1: {train_f1_weighted:.4f}, '
              f'Train Precision: {train_prec:.4f}, Train Recall: {train_rec:.4f}')
        
        print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val Macro-F1: {val_f1_macro:.4f}, '
              f'Val Micro-F1: {val_f1_micro:.4f}, Val Weighted-F1: {val_f1_weighted:.4f}, '
              f'Val Precision: {val_prec:.4f}, Val Recall: {val_rec:.4f}')
        
        if val_acc == 1.0:
            print("Early Stopping ... ")
            break
        
        # Save the best model based on validation F1 score
        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print("Saved best model")
    
    return val_loss, val_acc, val_f1_macro, val_f1_micro, val_f1_weighted, val_prec, val_rec

## ESC-50

In [ ]:
path = "/kaggle/input/environmental-sound-classification-50/audio/audio/"
df = pd.read_csv("/kaggle/input/environmental-sound-classification-50/esc50.csv")
df.head()

In [ ]:
res = {}

In [ ]:
c=0
for i in range(0,10):
    t = len(df[df['classID'] == i])
    print(i,":",t)
    c += t

In [ ]:
for f in range(0,5):
    
    model = MultiAudioClassifier(num_labels=50)
    model = model.to(device)
    
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=0.001)
    
    
    file_list2 = []
    val_list = []
    for d in [j for j in range(0,5) if j != f+1]:
        df1 = df[df["fold"] == d+1]
        for i in range(0, len(df1)):
            file_list2.append((tuple([i]),0))
            file_list2.append((tuple([i]),1))
            file_list2.append((tuple([i]),2))
            file_list2.append((tuple([i]),3))
            file_list2.append((tuple([i]),4))
            file_list2.append((tuple([i]),5))
            
            
    df1 = df[df["fold"] == f+1]
    for i in range(0, len(df1)):
        val_list.append((tuple([i]),0))
        val_list.append((tuple([i]),1))
        val_list.append((tuple([i]),2))
        val_list.append((tuple([i]),3))
        val_list.append((tuple([i]),4))
        val_list.append((tuple([i]),5))
    
    
    num_labels = 50
    train_ds = SoundDS(df,extractor,path,16000,file_list2,num_labels)
    val_ds = SoundDS(df,extractor,path,16000,val_list,num_labels)

    
    print(f"\nFold : {f + 1}\n")
    train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=16, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=64, shuffle=True, num_workers=16, pin_memory=True) 
    
    num_epochs = 20
    vl,va,vf_mac,vf_mic,vf_wei,vp,vr = train(model, train_dl, val_dl, criterion, optimizer, num_epochs, device, num_labels)
    res[f+1] = (vl,va,vf_mac,vf_mic,vf_wei,vp,vr)

In [ ]:
torch.save(model.state_dict(), 'final_model_esc50_mono.pth')

## US8K

In [26]:
# Second mix function for US8k
def mix(z,df,num_labels,path):
    audio = load_audio(path +"/"+"fold"+ str(df.iloc[z[0]]["fold"])+"/"+df.iloc[z[0]]['slice_file_name'])
    a = torch.zeros(num_labels)
    a[df.iloc[z[0]]["classID"]] = 1
    
#     for j in z[1:]:
#         audio += load_audio(path +"/"+"fold"+ str(df.iloc[j]["fold"])+"/"+df.iloc[j]['slice_file_name'])
#         a[df.iloc[j]["classID"]] = 1
        
    return audio,a

In [27]:
df = pd.read_csv("/kaggle/input/urbansound8k/UrbanSound8K.csv")
path = "/kaggle/input/urbansound8k"

In [ ]:
n = 10
for f in range(1,n+1): print(f, [j for j in range(1,n+1) if j != f])

In [ ]:
res = {}

In [ ]:
n  = 10
for f in range(1,n+1):
    
    model = MultiAudioClassifier(num_labels=10)
    model = model.to(device)
    
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    
    file_list2 = []
    val_list = []
    for d in [j for j in range(1,n+1) if j != f]:
        df1 = df[df["fold"] == d]
        for i in range(0, len(df1)):
            file_list2.append((tuple([i]),0))

            
    df1 = df[df["fold"] == f]
    for i in range(0, len(df1)):
        val_list.append((tuple([i]),0))

    
    num_labels = 10
    train_ds = SoundDS(df,extractor,path,16000,file_list2,num_labels)
    val_ds = SoundDS(df,extractor,path,16000,val_list,num_labels)

    print(f"\nFold : {f}\n")
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=16, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=32, shuffle=True, num_workers=16, pin_memory=True) 
    num_epochs = 20
    
    vl,va,vf_mac,vf_mic,vf_wei,vp,vr = train(model, train_dl, val_dl, criterion, optimizer, num_epochs, device, num_labels)
    res[f+1] = (vl,va,vf_mac,vf_mic,vf_wei,vp,vr)


In [28]:
res_aug = {}

In [ ]:
n  = 10
for f in range(1,6):
    
    model = MultiAudioClassifier(num_labels=10)
    model = model.to(device)
    
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    
    file_list2 = []
    val_list = []
    for d in [j for j in range(1,n+1) if j != f]:
        df1 = df[df["fold"] == d]
        for i in range(0, len(df1)):
            file_list2.append((tuple([i]),0))
            file_list2.append((tuple([i]),1))
            file_list2.append((tuple([i]),2))
            file_list2.append((tuple([i]),3))
            file_list2.append((tuple([i]),4))
            file_list2.append((tuple([i]),5))
            
    df1 = df[df["fold"] == f]
    for i in range(0, len(df1)):
        val_list.append((tuple([i]),0))
        val_list.append((tuple([i]),1))
        val_list.append((tuple([i]),2))
        val_list.append((tuple([i]),3))
        val_list.append((tuple([i]),4))
        val_list.append((tuple([i]),5))
    
    num_labels = 10
    train_ds = SoundDS(df,extractor,path,16000,file_list2,num_labels)
    val_ds = SoundDS(df,extractor,path,16000,val_list,num_labels)

    print(f"\nFold : {f + 1}\n")
    train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=16, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=256, shuffle=True, num_workers=16, pin_memory=True) 
    num_epochs = 20
    
    vl,va,vf_mac,vf_mic,vf_wei,vp,vr = train(model, train_dl, val_dl, criterion, optimizer, num_epochs, device, num_labels)
    res_aug[f+1] = (vl,va,vf_mac,vf_mic,vf_wei,vp,vr)


In [29]:
n  = 10
for f in range(2,6):
    
    model = MultiAudioClassifier(num_labels=10)
    model = model.to(device)
    
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    
    file_list2 = []
    val_list = []
    for d in [j for j in range(1,n+1) if j != f]:
        df1 = df[df["fold"] == d]
        for i in range(0, len(df1)):
            file_list2.append((tuple([i]),0))
            file_list2.append((tuple([i]),1))
            file_list2.append((tuple([i]),2))
            file_list2.append((tuple([i]),3))
            file_list2.append((tuple([i]),4))
            file_list2.append((tuple([i]),5))
            
    df1 = df[df["fold"] == f]
    for i in range(0, len(df1)):
        val_list.append((tuple([i]),0))
        val_list.append((tuple([i]),1))
        val_list.append((tuple([i]),2))
        val_list.append((tuple([i]),3))
        val_list.append((tuple([i]),4))
        val_list.append((tuple([i]),5))
    
    num_labels = 10
    train_ds = SoundDS(df,extractor,path,16000,file_list2,num_labels)
    val_ds = SoundDS(df,extractor,path,16000,val_list,num_labels)

    print(f"\nFold : {f + 1}\n")
    train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=16, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=256, shuffle=True, num_workers=16, pin_memory=True) 
    num_epochs = 20
    
    vl,va,vf_mac,vf_mic,vf_wei,vp,vr = train(model, train_dl, val_dl, criterion, optimizer, num_epochs, device, num_labels)
    res_aug[f+1] = (vl,va,vf_mac,vf_mic,vf_wei,vp,vr)


Using 2 GPUs

Fold : 3

Epoch 1/20


Validation: 100%|██████████| 21/21 [01:49<00:00,  5.23s/it]


Train Loss: 1.1038, Train Acc: 0.8347, Train Macro-F1: 0.6823, Train Micro-F1: 0.8347, Train Weighted-F1: 0.8285, Train Precision: 0.8347, Train Recall: 0.8347
Val Loss: 0.4817, Val Acc: 0.9454, Val Macro-F1: 0.7927, Val Micro-F1: 0.9454, Val Weighted-F1: 0.9393, Val Precision: 0.9454, Val Recall: 0.9454
Saved best model
Epoch 2/20


Validation: 100%|██████████| 21/21 [01:53<00:00,  5.40s/it]


Train Loss: 0.3409, Train Acc: 0.9559, Train Macro-F1: 0.8293, Train Micro-F1: 0.9559, Train Weighted-F1: 0.9510, Train Precision: 0.9559, Train Recall: 0.9559
Val Loss: 0.1783, Val Acc: 0.9794, Val Macro-F1: 0.9070, Val Micro-F1: 0.9794, Val Weighted-F1: 0.9765, Val Precision: 0.9794, Val Recall: 0.9794
Saved best model
Epoch 3/20


...

Train Loss: 0.0153, Train Acc: 0.9996, Train Macro-F1: 0.9995, Train Micro-F1: 0.9996, Train Weighted-F1: 0.9996, Train Precision: 0.9996, Train Recall: 0.9996
Val Loss: 0.0076, Val Acc: 0.9998, Val Macro-F1: 0.9998, Val Micro-F1: 0.9998, Val Weighted-F1: 0.9998, Val Precision: 0.9998, Val Recall: 0.9998
Epoch 9/20


Validation: 100%|██████████| 22/22 [01:50<00:00,  5.02s/it]


Train Loss: 0.0121, Train Acc: 0.9997, Train Macro-F1: 0.9997, Train Micro-F1: 0.9997, Train Weighted-F1: 0.9997, Train Precision: 0.9997, Train Recall: 0.9997
Val Loss: 0.0064, Val Acc: 0.9998, Val Macro-F1: 0.9999, Val Micro-F1: 0.9998, Val Weighted-F1: 0.9998, Val Precision: 0.9998, Val Recall: 0.9998
Epoch 10/20


Training:   0%|          | 0/183 [00:10<?, ?it/s]


KeyboardInterrupt: 

In [31]:
torch.save(model.state_dict(), 'final_model_US8K_mono.pth')

In [86]:
model = MultiAudioClassifier(10)
model = nn.DataParallel(model)

In [88]:
model.load_state_dict(torch.load("/kaggle/working/final_model_US8K_mono.pth"))

<All keys matched successfully>

In [ ]:
model.cuda()

In [89]:
pytorch_total_params = sum(p.numel() for p in model.parameters())

In [90]:
pytorch_total_params 

91155097

In [91]:
pytorch_total_params / (1e6)

91.155097

In [92]:
y = load_audio("/kaggle/input/testaudio2/Pitbull barking.mp3")

In [93]:
y = librosa.util.normalize(y)

In [94]:
f = extractor(y,sampling_rate=16000)['input_values'][0]

In [98]:
p = 0
model.eval()
with torch.no_grad():
    p = model(torch.Tensor(f).unsqueeze(0).cuda())

In [99]:
torch.argmax(p)

tensor(3, device='cuda:0')